In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report

import xgboost as xgb
import optuna

In [2]:
train_df = pd.read_csv("training_kerala(2003-2023).csv")
test_df = pd.read_csv("test_kerala(2024-25).csv")

print(train_df.shape)
print(test_df.shape)

(5774701, 14)
(352940, 14)


In [3]:
FEATURES = [
    "CHLOR_A",
    "day_sin",
    "day_cos",
    "month_sin",
    "month_cos",
    "LAT_scaled",
    "LON_scaled"
]

TARGET = "PHYTOBLOOM"

In [4]:
X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

In [8]:

import joblib

final_model =  joblib.load('lgb_model_Kerala.pkl')

final_model.fit(
    X_train,
    y_train
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.035221 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 907
[LightGBM] [Info] Number of data points in the train set: 5774701, number of used features: 7
[LightGBM] [Info] Start training from score -0.070857
[LightGBM] [Info] Start training from score -3.417544
[LightGBM] [Info] Start training from score -3.335066
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spl

,num_leaves,109
,max_depth,4
,learning_rate,0.043381244467624716
,n_estimators,130
,objective,'multiclass'
,min_split_gain,0.0876499877099334
,min_child_samples,33
,subsample,0.8917053192257713
,colsample_bytree,0.9642181939482745
,reg_alpha,0.044049787119522996
,reg_lambda,0.05299143759204272


In [9]:
y_pred1 = final_model.predict(
    X_train
)
y_pred = final_model.predict(
    X_test
)

In [10]:
print("TRAIN RESULTS")
print(
    classification_report(
        y_train,
        y_pred1
    )
)

macro_f1 = f1_score(
    y_train,
    y_pred1,
    average="macro"
)

print(
    "Train Macro F1:",
    macro_f1
)
print()
print("TEST RESULTS")
print(
    classification_report(
        y_test,
        y_pred
    )
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

print(
    "Test Macro F1:",
    macro_f1
)

TRAIN RESULTS
              precision    recall  f1-score   support

         0.0       1.00      0.99      0.99   5379682
         1.0       0.82      0.87      0.85    189369
         2.0       0.84      0.82      0.83    205650

    accuracy                           0.98   5774701
   macro avg       0.89      0.90      0.89   5774701
weighted avg       0.98      0.98      0.98   5774701

Train Macro F1: 0.8908372272386195

TEST RESULTS
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99    329160
         1.0       0.75      0.89      0.82     12313
         2.0       0.88      0.72      0.79     11467

    accuracy                           0.98    352940
   macro avg       0.88      0.87      0.87    352940
weighted avg       0.98      0.98      0.98    352940

Test Macro F1: 0.8675099387737455


In [60]:
def objective(trial):

    # params = {

    #     "objective": "multi:softprob",
    #     "num_class": 3,

    #     "n_estimators":
    #         trial.suggest_int(
    #             "n_estimators",
    #             100,
    #             500,
    #             step=50
    #         ),

    #     "max_depth":
    #         trial.suggest_int(
    #             "max_depth",
    #             4,
    #             12
    #         ),

    #     "learning_rate":
    #         trial.suggest_float(
    #             "learning_rate",
    #             0.01,
    #             0.3,
    #             log=True
    #         ),

    #     "subsample":
    #         trial.suggest_float(
    #             "subsample",
    #             0.6,
    #             1.0
    #         ),

    #     "colsample_bytree":
    #         trial.suggest_float(
    #             "colsample_bytree",
    #             0.6,
    #             1.0
    #         ),

    #     "min_child_weight":
    #         trial.suggest_int(
    #             "min_child_weight",
    #             1,
    #             20
    #         ),

    #     "gamma":
    #         trial.suggest_float(
    #             "gamma",
    #             1e-5,
    #             1.0,
    #             log=True
    #         ),

    #     "lambda":
    #         trial.suggest_float(
    #             "lambda",
    #             1e-3,
    #             10,
    #             log=True
    #         ),

    #     "alpha":
    #         trial.suggest_float(
    #             "alpha",
    #             1e-3,
    #             5,
    #             log=True
    #         ),

    #     "grow_policy":
    #         trial.suggest_categorical(
    #             "grow_policy",
    #             ["depthwise", "lossguide"]
    #         ),

    #     "tree_method": "hist",

    #     "random_state": 42,

    #     "n_jobs": -1
    # }

    params = {

    "objective": "multi:softprob",
    "num_class": 3,

    "n_estimators":
        trial.suggest_int(
            "n_estimators",
            5,
            100,
            step=5
        ),

    "max_depth":
        trial.suggest_int(
            "max_depth",
            3,
            15
        ),

    "learning_rate":
        trial.suggest_float(
            "learning_rate",
            0.05,
            0.30,
            log=True
        ),

    "subsample":
        trial.suggest_float(
            "subsample",
            0.60,
            1.00
        ),

    "colsample_bytree":
        trial.suggest_float(
            "colsample_bytree",
            0.70,
            1.00
        ),

    "min_child_weight":
        trial.suggest_int(
            "min_child_weight",
            13,
            28
        ),

    "gamma":
        trial.suggest_float(
            "gamma",
            1e-3,
            2.0,
            log=True
        ),

    "lambda":
        trial.suggest_float(
            "lambda",
            0.1,
            10,
            log=True
        ),

    "alpha":
        trial.suggest_float(
            "alpha",
            1.5,
            3.0
        ),

    "grow_policy":
        trial.suggest_categorical(
            "grow_policy",
            ["depthwise", "lossguide"]
        ),

    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1
    }
    # params = {

    # "objective": "multi:softprob",
    # "num_class": 3,

    # "n_estimators":
    #     trial.suggest_int(
    #         "n_estimators",
    #         25,
    #         70,
    #         step=5
    #     ),

    # "max_depth":
    #     trial.suggest_int(
    #         "max_depth",
    #         6,
    #         9
    #     ),

    # "learning_rate":
    #     trial.suggest_float(
    #         "learning_rate",
    #         0.18,
    #         0.28
    #     ),

    # "subsample":
    #     trial.suggest_float(
    #         "subsample",
    #         0.65,
    #         0.80
    #     ),

    # "colsample_bytree":
    #     trial.suggest_float(
    #         "colsample_bytree",
    #         0.75,
    #         0.90
    #     ),

    # "min_child_weight":
    #     trial.suggest_int(
    #         "min_child_weight",
    #         18,
    #         24
    #     ),

    # "gamma":
    #     trial.suggest_float(
    #         "gamma",
    #         0.02,
    #         0.25,
    #         log=True
    #     ),

    # "lambda":
    #     trial.suggest_float(
    #         "lambda",
    #         1.0,
    #         5.0,
    #         log=True
    #     ),

    # "alpha":
    #     trial.suggest_float(
    #         "alpha",
    #         1.8,
    #         3.0
    #     ),

    # "grow_policy":
    #     trial.suggest_categorical(
    #         "grow_policy",
    #         ["depthwise", "lossguide"]
    #     ),

    # "tree_method": "hist",
    # "random_state": 42,
    # "n_jobs": -1
    # }


    skf = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

    scores = []

    for train_idx, val_idx in skf.split(X_train,y_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = xgb.XGBClassifier(
            **params
        )

        model.fit(
            X_tr,
            y_tr,
            verbose=False
        )

        pred = model.predict(X_val)

        macro_f1 = f1_score(
            y_val,
            pred,
            average="macro"
        )

        scores.append(
            macro_f1
        )

    return np.mean(scores)

In [61]:
study5 = optuna.create_study(direction="maximize")

study5.optimize(
    objective,
    n_trials=100,
    show_progress_bar=True
)

[I 2026-07-08 11:09:07,189] A new study created in memory with name: no-name-af3d633c-946e-42e3-b8e0-899599af9df8


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-07-08 11:14:55,008] Trial 0 finished with value: 0.8860829450489301 and parameters: {'n_estimators': 40, 'max_depth': 4, 'learning_rate': 0.09105352399516183, 'subsample': 0.7835041239315401, 'colsample_bytree': 0.7899046976697484, 'min_child_weight': 19, 'gamma': 0.028755650052503207, 'lambda': 0.7828529458810957, 'alpha': 2.7407144877324434, 'grow_policy': 'depthwise'}. Best is trial 0 with value: 0.8860829450489301.
[I 2026-07-08 11:35:14,398] Trial 1 finished with value: 0.925395670739813 and parameters: {'n_estimators': 60, 'max_depth': 15, 'learning_rate': 0.1801566547415575, 'subsample': 0.9488942895420532, 'colsample_bytree': 0.7542247851705182, 'min_child_weight': 15, 'gamma': 0.20495092512226096, 'lambda': 3.8763651840296447, 'alpha': 1.598665039623781, 'grow_policy': 'lossguide'}. Best is trial 1 with value: 0.925395670739813.
[I 2026-07-08 11:41:47,629] Trial 2 finished with value: 0.8896877297729908 and parameters: {'n_estimators': 40, 'max_depth': 5, 'learning_rat

In [62]:
study5.best_params

{'n_estimators': 85,
 'max_depth': 15,
 'learning_rate': 0.29889196032214477,
 'subsample': 0.9233958796791396,
 'colsample_bytree': 0.8802508856825477,
 'min_child_weight': 18,
 'gamma': 0.10280301365476707,
 'lambda': 2.0984819414361886,
 'alpha': 1.8440065348785262,
 'grow_policy': 'depthwise'}

In [52]:
study4 = optuna.create_study(direction="maximize")

study4.optimize(
    objective,
    n_trials=40,
    show_progress_bar=True
)

[I 2026-07-07 16:03:14,532] A new study created in memory with name: no-name-d25a822c-cfc8-4a7f-9484-489c2e2eb3fb


  0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-07-07 16:05:43,776] Trial 0 finished with value: 0.9009784194949976 and parameters: {'n_estimators': 50, 'max_depth': 6, 'learning_rate': 0.22809759829945186, 'subsample': 0.6878618748580405, 'colsample_bytree': 0.8346732997470656, 'min_child_weight': 19, 'gamma': 0.19278900503808263, 'lambda': 3.2160840089276763, 'alpha': 2.1200113333332333, 'grow_policy': 'depthwise'}. Best is trial 0 with value: 0.9009784194949976.
[I 2026-07-07 16:08:32,643] Trial 1 finished with value: 0.9081641526832362 and parameters: {'n_estimators': 55, 'max_depth': 8, 'learning_rate': 0.20349739130584468, 'subsample': 0.7639678305405385, 'colsample_bytree': 0.8649883533043743, 'min_child_weight': 24, 'gamma': 0.025610462481654356, 'lambda': 4.6243401186460575, 'alpha': 2.4671578236796865, 'grow_policy': 'lossguide'}. Best is trial 1 with value: 0.9081641526832362.
[I 2026-07-07 16:12:08,173] Trial 2 finished with value: 0.9094401664749843 and parameters: {'n_estimators': 65, 'max_depth': 7, 'learning_

In [53]:
study4.best_params

{'n_estimators': 65,
 'max_depth': 9,
 'learning_rate': 0.27845683489755757,
 'subsample': 0.7379678337089801,
 'colsample_bytree': 0.824730578898164,
 'min_child_weight': 18,
 'gamma': 0.16201766936019735,
 'lambda': 1.2549769476643777,
 'alpha': 2.2957934725709035,
 'grow_policy': 'lossguide'}

In [7]:
study = optuna.create_study(direction="maximize")

study.optimize(
    objective,
    n_trials=20,
    show_progress_bar=True
)

[I 2026-07-06 18:01:52,358] A new study created in memory with name: no-name-7db6a485-1b9b-4298-b246-ffd1481f9a55


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-07-06 18:38:22,343] Trial 0 finished with value: 0.9263274075645759 and parameters: {'n_estimators': 450, 'max_depth': 10, 'learning_rate': 0.07743187468424283, 'subsample': 0.897258043268682, 'colsample_bytree': 0.9626412064852917, 'min_child_weight': 10, 'gamma': 1.245822935014722e-05, 'lambda': 0.18654366426950392, 'alpha': 4.612219923536247, 'grow_policy': 'lossguide'}. Best is trial 0 with value: 0.9263274075645759.
[I 2026-07-06 19:03:53,924] Trial 1 finished with value: 0.9104363817753593 and parameters: {'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.04976578745175034, 'subsample': 0.8934264771817089, 'colsample_bytree': 0.9759189939027002, 'min_child_weight': 11, 'gamma': 0.0254687703348067, 'lambda': 0.2991776548705129, 'alpha': 0.1135291841515351, 'grow_policy': 'depthwise'}. Best is trial 0 with value: 0.9263274075645759.
[I 2026-07-06 19:14:12,363] Trial 2 finished with value: 0.9097649198921622 and parameters: {'n_estimators': 200, 'max_depth': 5, 'learni

In [8]:
study.best_params

{'n_estimators': 450,
 'max_depth': 9,
 'learning_rate': 0.28653029528952517,
 'subsample': 0.8299399172531643,
 'colsample_bytree': 0.999020831073057,
 'min_child_weight': 15,
 'gamma': 1.0793730751765798e-05,
 'lambda': 0.1444310996476045,
 'alpha': 1.4062209879985144,
 'grow_policy': 'lossguide'}

In [28]:
study2 = optuna.create_study(direction="maximize")

study2.optimize(
    objective,
    n_trials=20,
    show_progress_bar=True
)

[I 2026-07-07 13:37:46,452] A new study created in memory with name: no-name-cd41294a-bda8-41c0-b2c6-d6068cec8593


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-07-07 13:39:23,999] Trial 0 finished with value: 0.8939566383436566 and parameters: {'n_estimators': 40, 'max_depth': 6, 'learning_rate': 0.114195564624584, 'subsample': 0.9888607241204561, 'colsample_bytree': 0.7785965333137245, 'min_child_weight': 19, 'gamma': 0.0011545503781552771, 'lambda': 0.9367564781468635, 'alpha': 2.8516804676993246, 'grow_policy': 'depthwise'}. Best is trial 0 with value: 0.8939566383436566.
[I 2026-07-07 13:40:20,037] Trial 1 finished with value: 0.8971591946625462 and parameters: {'n_estimators': 20, 'max_depth': 8, 'learning_rate': 0.1400101537721588, 'subsample': 0.8971178455925676, 'colsample_bytree': 0.9040322938506244, 'min_child_weight': 17, 'gamma': 0.3159556918689311, 'lambda': 6.5388793259545785, 'alpha': 2.973996329231856, 'grow_policy': 'depthwise'}. Best is trial 1 with value: 0.8971591946625462.
[I 2026-07-07 13:42:26,884] Trial 2 finished with value: 0.907456261009717 and parameters: {'n_estimators': 50, 'max_depth': 7, 'learning_rate'

In [29]:
study2.best_params

{'n_estimators': 45,
 'max_depth': 8,
 'learning_rate': 0.23852327197668097,
 'subsample': 0.7076673436861586,
 'colsample_bytree': 0.8463098832287198,
 'min_child_weight': 20,
 'gamma': 0.07447176749014245,
 'lambda': 2.583648477234089,
 'alpha': 2.2807456988396533,
 'grow_policy': 'depthwise'}

In [43]:
study3 = optuna.create_study(direction="maximize")

study3.optimize(
    objective,
    n_trials=20,
    show_progress_bar=True
)

[I 2026-07-07 15:04:55,514] A new study created in memory with name: no-name-3def8f8e-a7a6-47f1-be51-72c785b85f41


  0%|          | 0/20 [00:00<?, ?it/s]

d:\miniconda3\envs\incois\Lib\site-packages\optuna\distributions.py:684: UserWarning: The distribution is specified by [5, 70] and step=2, but the range is not divisible by `step`. It will be replaced with [5, 69].
  optuna_warn(


[I 2026-07-07 15:07:35,557] Trial 0 finished with value: 0.8815355266978344 and parameters: {'n_estimators': 51, 'max_depth': 3, 'learning_rate': 0.07423325893948823, 'subsample': 0.8852823657170523, 'colsample_bytree': 0.7895142645356187, 'min_child_weight': 24, 'gamma': 0.007429431825583001, 'lambda': 7.897382091254342, 'alpha': 1.5528038674510733, 'grow_policy': 'lossguide'}. Best is trial 0 with value: 0.8815355266978344.


d:\miniconda3\envs\incois\Lib\site-packages\optuna\distributions.py:684: UserWarning: The distribution is specified by [5, 70] and step=2, but the range is not divisible by `step`. It will be replaced with [5, 69].
  optuna_warn(


...

[I 2026-07-07 15:52:00,794] Trial 17 finished with value: 0.9028446675398711 and parameters: {'n_estimators': 21, 'max_depth': 10, 'learning_rate': 0.15035170211637178, 'subsample': 0.8301396600717824, 'colsample_bytree': 0.8102627546133767, 'min_child_weight': 26, 'gamma': 0.5806346680141454, 'lambda': 0.5389677180492017, 'alpha': 2.6399980526392373, 'grow_policy': 'depthwise'}. Best is trial 14 with value: 0.9141698987153857.


d:\miniconda3\envs\incois\Lib\site-packages\optuna\distributions.py:684: UserWarning: The distribution is specified by [5, 70] and step=2, but the range is not divisible by `step`. It will be replaced with [5, 69].
  optuna_warn(


[I 2026-07-07 15:54:53,398] Trial 18 finished with value: 0.91074292469038 and parameters: {'n_estimators': 53, 'max_depth': 8, 'learning_rate': 0.26961393330490857, 'subsample': 0.7346118185109927, 'colsample_bytree': 0.7319448775500784, 'min_child_weight': 23, 'gamma': 0.06742590109984183, 'lambda': 0.5414739911107359, 'alpha': 2.811722153254741, 'grow_policy': 'depthwise'}. Best is trial 14 with value: 0.9141698987153857.


d:\miniconda3\envs\incois\Lib\site-packages\optuna\distributions.py:684: UserWarning: The distribution is specified by [5, 70] and step=2, but the range is not divisible by `step`. It will be replaced with [5, 69].
  optuna_warn(


[I 2026-07-07 15:58:00,525] Trial 19 finished with value: 0.9127169418335785 and parameters: {'n_estimators': 63, 'max_depth': 9, 'learning_rate': 0.1978454597399138, 'subsample': 0.8368866837077272, 'colsample_bytree': 0.7921421519914678, 'min_child_weight': 26, 'gamma': 0.7682538579914926, 'lambda': 0.16771971208015835, 'alpha': 2.543684874225817, 'grow_policy': 'depthwise'}. Best is trial 14 with value: 0.9141698987153857.


In [44]:
study3.best_params

{'n_estimators': 63,
 'max_depth': 10,
 'learning_rate': 0.16861870260597978,
 'subsample': 0.7510999176716665,
 'colsample_bytree': 0.748876036957902,
 'min_child_weight': 26,
 'gamma': 0.4705490772217076,
 'lambda': 0.24992638565878145,
 'alpha': 2.71055676813654,
 'grow_policy': 'depthwise'}

In [9]:
fig = optuna.visualization.plot_optimization_history(study)
fig.show()

In [10]:
optuna.visualization.plot_param_importances(study)

In [108]:
def objective_multi(trial):
    # params = {
    #     "objective": "multi:softprob",
    #     "num_class": 3,

    #     "n_estimators": trial.suggest_int("n_estimators", 20, 90, step=5),
    #     "max_depth": trial.suggest_int("max_depth", 4, 9),
    #     "learning_rate": trial.suggest_float("learning_rate", 0.08, 0.25, log=True),
    #     "subsample": trial.suggest_float("subsample", 0.65, 0.85),
    #     "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 0.88),
    #     "min_child_weight": trial.suggest_int("min_child_weight", 15, 30),
    #     "gamma": trial.suggest_float("gamma", 0.02, 1.0, log=True),
    #     "lambda": trial.suggest_float("lambda", 0.3, 5.0, log=True),
    #     "alpha": trial.suggest_float("alpha", 1.6, 2.8),
    #     "grow_policy": trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"]),

    #     "tree_method": "hist",
    #     "random_state": 42,
    #     "n_jobs": -1,
    # }
    params = {
        "objective": "multi:softprob",
        "num_class": 3,

        "n_estimators": trial.suggest_int("n_estimators", 30, 90, step=5),
        "max_depth": trial.suggest_int("max_depth", 3, 5),  # narrowed hard around 4
        "learning_rate": trial.suggest_float("learning_rate", 0.10, 0.24, log=True),
        "subsample": trial.suggest_float("subsample", 0.70, 0.85),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 0.85),
        "min_child_weight": trial.suggest_int("min_child_weight", 18, 30),
        "gamma": trial.suggest_float("gamma", 0.05, 0.5, log=True),
        "lambda": trial.suggest_float("lambda", 0.4, 2.5, log=True),
        "alpha": trial.suggest_float("alpha", 1.7, 2.8),
        "grow_policy": "depthwise",

        "tree_method": "hist",
        "random_state": 42,
        "n_jobs": -1,
    }


    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    gaps, val_scores = [], []

    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = xgb.XGBClassifier(**params)
        model.fit(X_tr, y_tr, verbose=False)

        train_f1 = f1_score(y_tr, model.predict(X_tr), average="macro")
        val_f1 = f1_score(y_val, model.predict(X_val), average="macro")

        val_scores.append(val_f1)
        gaps.append(train_f1 - val_f1)

    mean_val = np.mean(val_scores)
    mean_gap = np.mean(gaps)

    # Return a tuple: optuna will treat this as two objectives
    return mean_val, mean_gap

In [73]:
study = optuna.create_study(directions=["maximize", "minimize"]) 
study.optimize(objective_multi, n_trials=40, show_progress_bar=True)

[I 2026-07-09 10:47:36,173] A new study created in memory with name: no-name-5c9297cb-cd5b-4c68-b761-2e54470d9ce0


  0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-07-09 10:55:34,235] Trial 0 finished with values: [0.8940831119882159, 0.0002878500748883184] and parameters: {'n_estimators': 90, 'max_depth': 4, 'learning_rate': 0.14513877954983637, 'subsample': 0.8243606105581088, 'colsample_bytree': 0.8435126520788934, 'min_child_weight': 15, 'gamma': 0.23075042476548593, 'lambda': 0.42541463394045087, 'alpha': 2.177328895220446, 'grow_policy': 'lossguide'}.
[I 2026-07-09 11:01:56,814] Trial 1 finished with values: [0.9005749617612743, 0.001431913633477988] and parameters: {'n_estimators': 35, 'max_depth': 8, 'learning_rate': 0.11070378606860946, 'subsample': 0.6900466524455382, 'colsample_bytree': 0.872477862649814, 'min_child_weight': 17, 'gamma': 0.06830778465916278, 'lambda': 0.37261076069590915, 'alpha': 2.5441165431528012, 'grow_policy': 'lossguide'}.
[I 2026-07-09 11:09:14,935] Trial 2 finished with values: [0.8882894478740511, 0.00021281536956159198] and parameters: {'n_estimators': 55, 'max_depth': 4, 'learning_rate': 0.0988491251

In [75]:
best_trials = study.best_trials
i=0

for t in best_trials:
    print(f"{i} val_f1={t.values[0]:.4f}, gap={t.values[1]:.4f}, params={t.params}")
    i+=1

0 val_f1=0.8883, gap=0.0002, params={'n_estimators': 55, 'max_depth': 4, 'learning_rate': 0.09884912516607705, 'subsample': 0.8170500490138183, 'colsample_bytree': 0.7834112629406853, 'min_child_weight': 23, 'gamma': 0.07218612608497117, 'lambda': 1.080719113507053, 'alpha': 2.1326477648431625, 'grow_policy': 'lossguide'}
1 val_f1=0.9040, gap=0.0014, params={'n_estimators': 80, 'max_depth': 6, 'learning_rate': 0.1870031942127432, 'subsample': 0.7514880011101248, 'colsample_bytree': 0.8117899459989935, 'min_child_weight': 28, 'gamma': 0.09068057779350967, 'lambda': 1.7937098412164916, 'alpha': 1.7685666288956698, 'grow_policy': 'depthwise'}
2 val_f1=0.9139, gap=0.0041, params={'n_estimators': 85, 'max_depth': 9, 'learning_rate': 0.1543702857014121, 'subsample': 0.6778220005370434, 'colsample_bytree': 0.7999259891783939, 'min_child_weight': 16, 'gamma': 0.16616573164982693, 'lambda': 3.838577464451175, 'alpha': 1.7587614063047896, 'grow_policy': 'lossguide'}
3 val_f1=0.9070, gap=0.0018, 

In [1]:
params=best_trials[12].params

NameError: name 'best_trials' is not defined

In [2]:
params={'n_estimators': 70,
 'max_depth': 4,
 'learning_rate': 0.2154777798466142,
 'subsample': 0.7750803324939916,
 'colsample_bytree': 0.7552877217759895,
 'min_child_weight': 21,
 'gamma': 0.24030240093852506,
 'lambda': 1.8687253559426629,
 'alpha': 2.6966546388895587,
 'grow_policy': 'depthwise'}


In [7]:
import xgboost as xgb

model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    **params,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train
)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.7552877217759895
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.dat

In [12]:
import joblib

joblib.dump(model, r"D:\INCOIS\Notebooks\xgb_model_Kerala.pkl")

['D:\\INCOIS\\Notebooks\\xgb_model_Kerala.pkl']

In [8]:
y_pred1 = model.predict(
    X_train
)

In [9]:
print(
    classification_report(
        y_train,
        y_pred1
    )
)

macro_f1 = f1_score(
    y_train,
    y_pred1,
    average="macro"
)

print(
    "Train Macro F1:",
    macro_f1
)

              precision    recall  f1-score   support

         0.0       1.00      0.99      0.99   5379682
         1.0       0.83      0.87      0.85    189369
         2.0       0.85      0.84      0.84    205650

    accuracy                           0.98   5774701
   macro avg       0.89      0.90      0.90   5774701
weighted avg       0.98      0.98      0.98   5774701

Train Macro F1: 0.8953717659135294


In [10]:
y_pred = model.predict(
    X_test
)

In [11]:
print(
    classification_report(
        y_test,
        y_pred
    )
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

print(
    "Test Macro F1:",
    macro_f1
)

              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99    329160
         1.0       0.77      0.88      0.82     12313
         2.0       0.88      0.72      0.79     11467

    accuracy                           0.98    352940
   macro avg       0.88      0.87      0.87    352940
weighted avg       0.98      0.98      0.98    352940

Test Macro F1: 0.8697377951192976


In [18]:
best_params=study.best_params
best_params

{'n_estimators': 450,
 'max_depth': 9,
 'learning_rate': 0.28653029528952517,
 'subsample': 0.8299399172531643,
 'colsample_bytree': 0.999020831073057,
 'min_child_weight': 15,
 'gamma': 1.0793730751765798e-05,
 'lambda': 0.1444310996476045,
 'alpha': 1.4062209879985144,
 'grow_policy': 'lossguide'}

In [17]:
from sklearn.metrics import f1_score
import copy
import xgboost as xgb
import pandas as pd

def tune_one_parameter(param_name, values):

    results = []

    for value in values:

        params = copy.deepcopy(best_params)
        params[param_name] = value

        model = xgb.XGBClassifier(
            objective="multi:softprob",
            num_class=3,
            tree_method="hist",
            random_state=42,
            n_jobs=-1,
            **params
        )

        model.fit(X_train, y_train)

        # Train prediction
        train_pred = model.predict(X_train)
        train_f1 = f1_score(
            y_train,
            train_pred,
            average="macro"
        )

        # Test prediction
        test_pred = model.predict(X_test)
        test_f1 = f1_score(
            y_test,
            test_pred,
            average="macro"
        )

        gap = train_f1 - test_f1

        results.append([value, train_f1, test_f1, gap])

        print(
            f"{param_name}={value} | "
            f"Train={train_f1:.4f} | "
            f"Test={test_f1:.4f} | "
            f"Gap={gap:.4f}"
        )

In [25]:
tune_one_parameter(
    "alpha",
    [0.32,0.52,1.23,1.73,2.11]
)

alpha=0.32 | Train=0.9529 | Test=0.8637 | Gap=0.0891
alpha=0.52 | Train=0.9529 | Test=0.8633 | Gap=0.0896
alpha=1.23 | Train=0.9534 | Test=0.8635 | Gap=0.0899
alpha=1.73 | Train=0.9529 | Test=0.8632 | Gap=0.0897
alpha=2.11 | Train=0.9529 | Test=0.8642 | Gap=0.0887


In [26]:
tune_one_parameter(
    "max_depth",
    [3,4,5,6,7,8,10]
)

max_depth=3 | Train=0.9089 | Test=0.8670 | Gap=0.0419
max_depth=4 | Train=0.9169 | Test=0.8672 | Gap=0.0497
max_depth=5 | Train=0.9245 | Test=0.8672 | Gap=0.0572
max_depth=6 | Train=0.9322 | Test=0.8664 | Gap=0.0658
max_depth=7 | Train=0.9397 | Test=0.8651 | Gap=0.0746
max_depth=8 | Train=0.9467 | Test=0.8648 | Gap=0.0818
max_depth=10 | Train=0.9593 | Test=0.8638 | Gap=0.0955


In [19]:
tune_one_parameter(
    "min_child_weight",
    [8,12,17,21]
)

min_child_weight=8 | Train=0.9579 | Test=0.8628 | Gap=0.0951
min_child_weight=12 | Train=0.9550 | Test=0.8634 | Gap=0.0916
min_child_weight=17 | Train=0.9521 | Test=0.8646 | Gap=0.0875
min_child_weight=21 | Train=0.9505 | Test=0.8629 | Gap=0.0876


In [20]:
tune_one_parameter(
    "min_child_weight",
    [16,18,19,20]
)

min_child_weight=16 | Train=0.9527 | Test=0.8647 | Gap=0.0880
min_child_weight=18 | Train=0.9519 | Test=0.8634 | Gap=0.0885
min_child_weight=19 | Train=0.9512 | Test=0.8631 | Gap=0.0882
min_child_weight=20 | Train=0.9506 | Test=0.8633 | Gap=0.0873


In [22]:
tune_one_parameter(
    "n_estimators",
    [200,400,500,550]
)

n_estimators=200 | Train=0.9407 | Test=0.8650 | Gap=0.0757
n_estimators=400 | Train=0.9515 | Test=0.8647 | Gap=0.0868
n_estimators=500 | Train=0.9552 | Test=0.8642 | Gap=0.0911
n_estimators=550 | Train=0.9567 | Test=0.8635 | Gap=0.0932


In [23]:
tune_one_parameter(
    "n_estimators",
    [20,40,100]
)

n_estimators=20 | Train=0.9095 | Test=0.8690 | Gap=0.0405
n_estimators=40 | Train=0.9158 | Test=0.8684 | Gap=0.0473
n_estimators=100 | Train=0.9299 | Test=0.8669 | Gap=0.0630


In [24]:
tune_one_parameter(
    "n_estimators",
    [5,10,15]
)

n_estimators=5 | Train=0.8969 | Test=0.8675 | Gap=0.0294
n_estimators=10 | Train=0.9024 | Test=0.8694 | Gap=0.0331
n_estimators=15 | Train=0.9067 | Test=0.8700 | Gap=0.0367
